In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import make_url

# read the latest value from .env
load_dotenv(override=True)

# checking the connection 
engine = create_engine(os.getenv("DATABASE_URL"), pool_pre_ping=True)

# project and server
url = make_url(os.getenv("DATABASE_URL"))
print("User:", url.username)
print("Host:", url.host)

# test 
with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar())
    print(conn.execute(text(
        "SELECT table_schema, table_name FROM information_schema.tables "
        "WHERE table_name = 'air_quality'")).fetchall())

User: postgres.apsdgrcvftokfxkebhin
Host: aws-0-ap-southeast-2.pooler.supabase.com
PostgreSQL 17.11 on aarch64-unknown-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit
[('public', 'air_quality')]


In [2]:
# re-upload the clean data (now with isolated PM10 spikes removed)
df = pd.read_csv("C:\\Users\\suraj\\Downloads\\henderson_clean_final_data.csv", parse_dates=["datetime"])
df.columns = [c.lower() for c in df.columns]
print(df.columns.tolist(), len(df))

# upload to a table called air_quality
df.to_sql("air_quality", engine, if_exists="replace", index=False,
          chunksize=5000, method="multi")

# replacing the table removes its key and security settings, so add them back
with engine.begin() as conn:
    conn.execute(text("ALTER TABLE air_quality ADD PRIMARY KEY (datetime)"))
    conn.execute(text("alter table air_quality enable row level security"))
    conn.execute(text('create policy "read only" on air_quality for select to anon using (true)'))
    conn.execute(text("grant select on air_quality to anon"))

# check
with engine.connect() as conn:
    print(conn.execute(text(
        "SELECT COUNT(*), MIN(datetime), MAX(datetime) FROM air_quality")).fetchone())

['datetime', 'no2', 'pm10', 'rh', 'temp', 'ws'] 150432
(150432, datetime.datetime(2006, 1, 1, 0, 0), datetime.datetime(2023, 2, 28, 23, 0))


In [3]:
# upload wind direction for the dashboard polar plots
raw = pd.read_excel(r"C:\Users\suraj\Downloads\1C. Dashboard_data_Henderson.xlsx", sheet_name="Main")

wd = raw[raw["Parameter"] == "WD"].copy()
wd["datetime"] = pd.to_datetime(wd["Date"]) + pd.to_timedelta(wd["Time"].astype(str))
wd = wd.drop_duplicates("datetime")[["datetime", "Value"]].rename(columns={"Value": "wd"})
print(len(wd), wd["datetime"].min(), "to", wd["datetime"].max())

wd.to_sql("wind_direction", engine, if_exists="replace", index=False,
          chunksize=5000, method="multi")

# key + read-only security
with engine.begin() as conn:
    conn.execute(text("ALTER TABLE wind_direction ADD PRIMARY KEY (datetime)"))
    conn.execute(text("alter table wind_direction enable row level security"))
    conn.execute(text('create policy "read only" on wind_direction for select to anon using (true)'))
    conn.execute(text("grant select on wind_direction to anon"))

with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*), MIN(datetime), MAX(datetime) FROM wind_direction")).fetchone())

4325 2022-09-01 00:00:00 to 2023-02-28 23:00:00
(4325, datetime.datetime(2022, 9, 1, 0, 0), datetime.datetime(2023, 2, 28, 23, 0))


In [28]:
# replace the predictions table with the new 80/20 test-set predictions
pred = pd.read_csv("C:\\Users\suraj\\Downloads\\predictions_test.csv", parse_dates=["datetime"])
print(pred.shape)

pred.to_sql("predictions", engine, if_exists="replace", index=False,
            chunksize=5000, method="multi")

# replacing the table removes its security settings, so turn them back on
with engine.begin() as conn:
    conn.execute(text("alter table predictions enable row level security"))
    conn.execute(text('create policy "read only" on predictions for select to anon using (true)'))
    conn.execute(text("grant select on predictions to anon"))

# check
with engine.connect() as conn:
    print(conn.execute(text(
        "SELECT pollutant, model, COUNT(*), MIN(datetime), MAX(datetime) "
        "FROM predictions GROUP BY pollutant, model")).fetchall())

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
C:\Users\suraj\AppData\Local\Temp\ipykernel_21200\2906226306.py:2: SyntaxWarning: invalid escape sequence '\s'
  pred = pd.read_csv("C:\\Users\suraj\\Downloads\\predictions_test.csv", parse_dates=["datetime"])


(50347, 5)
[('NO2', 'LightGBM', 24350, datetime.datetime(2017, 7, 15, 8, 0), datetime.datetime(2023, 2, 1, 0, 0)), ('PM10', 'LightGBM', 25997, datetime.datetime(2018, 7, 21, 5, 0), datetime.datetime(2023, 2, 1, 0, 0))]


In [30]:
#  upload the PM10 3-hour and 24-hour average forecasts for the dashboard
pred_avg = pd.read_csv("C:\\Users\\suraj\\Downloads\\predictions_pm10_avg.csv", parse_dates=["datetime"])
print(pred_avg.groupby("horizon_hours").size())

pred_avg.to_sql("predictions_pm10_avg", engine, if_exists="replace", index=False,
                chunksize=5000, method="multi")

# key + read-only security (replacing the table removes them)
with engine.begin() as conn:
    conn.execute(text("ALTER TABLE predictions_pm10_avg ADD PRIMARY KEY (datetime, horizon_hours)"))
    conn.execute(text("alter table predictions_pm10_avg enable row level security"))
    conn.execute(text('create policy "read only" on predictions_pm10_avg for select to anon using (true)'))
    conn.execute(text("grant select on predictions_pm10_avg to anon"))

with engine.connect() as conn:
    print(conn.execute(text(
        "SELECT horizon_hours, model, COUNT(*), MIN(datetime), MAX(datetime) "
        "FROM predictions_pm10_avg GROUP BY horizon_hours, model")).fetchall())

horizon_hours
3     27994
24    28970
dtype: int64
[(3, 'LightGBM (Huber)', 27994, datetime.datetime(2019, 6, 4, 18, 0), datetime.datetime(2023, 2, 28, 20, 0)), (24, 'LightGBM (Huber)', 28970, datetime.datetime(2019, 8, 7, 22, 0), datetime.datetime(2023, 2, 27, 23, 0))]


In [32]:
%pip install supabase

Note: you may need to restart the kernel to use updated packages.


In [34]:
from supabase import create_client

# read the API details from .env
load_dotenv(override=True)
sb = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

# test 1: read one day of data through the API
res = (sb.table("air_quality")
         .select("*")
         .gte("datetime", "2019-07-01")
         .lt("datetime", "2019-07-02")
         .order("datetime")
         .execute())

api_df = pd.DataFrame(res.data)
print("Rows from API:", len(api_df))
api_df.head()

Rows from API: 24


,datetime,no2,pm10,rh,temp,ws
0,2019-07-01T00:00:00,13.3,16.6,82.9,14,0.8
1,2019-07-01T01:00:00,11.7,12.2,74.8,11,0.4
2,2019-07-01T02:00:00,6.4,11.8,75.8,11,0.6
3,2019-07-01T03:00:00,8.9,13.6,76.3,10,0.6
4,2019-07-01T04:00:00,7.4,1.5,79.2,10,0.5


In [36]:
# test 2: check the API is read-only (this should FAIL)
try:
    sb.table("air_quality").insert({"datetime": "2030-01-01T00:00:00", "pm10": 1}).execute()
    print("WARNING: insert worked - security is not set correctly")
except Exception as e:
    print("Insert blocked (good):", str(e)[:120])

Insert blocked (good): {'message': 'new row violates row-level security policy for table "air_quality"', 'code': '42501', 'hint': None, 'detail
